In [1]:
%load_ext autoreload
%autoreload 2


from libthesis import pdf_writer, update_layout

In [2]:
import pandas
import plotly.express as px
import plotly.graph_objects as go
import json

# Plot config
species_colors = {
    "Chimpanzee":    "#CC0000",   # Bold Red (still legible for most CVD types)
    "Gorilla":    "#E69F00",  # Orange
    "Macaque": "#0072B2",  # Blue
}

symbol_sequence = {
    "has_cycles": "x",
    "no_cycles": "circle"
}


In [3]:
def graph_perf(k : int = None):

    # Load data
    df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")

    # Ensure we don't drop dict-typed columns (like madb_time) prematurely
    # Only drop rows where cogent3_time or score are missing
    df = df[
        df["cogent3_time"].notnull() &
        df["cogent3_score"].notnull() &
        df["madb_time"].notnull() &  # keep the nested dict
        df["madb_score"].notnull()
    ].copy()

    # Add cycle status for plotting
    df["cycle_status"] = df["madb_cycles"].map({True: "has_cycles", False: "no_cycles"})

    if k:
        # Only keep rows for desired kmer
        kmer = k
        df = df[df["kmer_size"] == kmer].copy()


    axis_range = (0, 0.2)

    # Filter to rows with valid times
    df_time = df[df["madb_time"].notnull()].copy()

    fig_time = px.scatter(
        df_time,
        x="cogent3_time",
        y="madb_time",
        color="species",
        color_discrete_map=species_colors,
        symbol="cycle_status",
        symbol_map=symbol_sequence,
        hover_data=["unique_id"]
    )

    fig_time.update_traces(marker=dict(size=10), showlegend=False)

    fig_time.add_trace(
        go.Scatter(
            x=axis_range,
            y=axis_range,
            mode="lines",
            line=dict(dash="dot", color="gray"),
            showlegend=False
        )
    )
    fig_time.add_annotation(
        text="Below line = MADB faster",
        xref="paper", yref="paper",
        x=0.98, y=0.02,
        xanchor="right", yanchor="bottom",
        font=dict(size=12),
        showarrow=False,
    )

    fig_time.update_layout(
        width=500,
        height=500,
        margin=dict(l=40, r=20, t=20, b=40),
        xaxis_title="Cogent3 execution (s)",
        yaxis_title=f"MADB execution (s)",
        xaxis=dict(range=axis_range, scaleanchor="y", scaleratio=1),
        yaxis=dict(range=axis_range),
    )

    update_layout(fig_time, in_panel=True, x_title="Needleman-Wunsch execution (s)", y_title="MADB execution (s)")

    fig_time.show()

    write_pdf = pdf_writer()
    write_pdf(fig_time, "madb_time_vs_cogent3_time")

graph_perf(15)

In [4]:
def graph_accuracy(k):

    # Extract MADB time for k=25 (assuming madb_time is still a stringified dict)
    import json
    import pandas

    # Load data
    df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")

    # Ensure we don't drop dict-typed columns (like madb_time) prematurely
    # Only drop rows where cogent3_time or score are missing
    df = df[
        df["cogent3_time"].notnull() &
        df["cogent3_score"].notnull() &
        df["madb_time"].notnull() &  # keep the nested dict
        df["madb_score"].notnull()
    ].copy()

    # Add cycle status for plotting
    df["cycle_status"] = df["madb_cycles"].map({True: "has_cycles", False: "no_cycles"})


    # Only keep rows for desired kmer
    kmer = k
    df = df[df["kmer_size"] == kmer].copy()

    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import r2_score
    import numpy

    # Regression inputs
    X = df["cogent3_score"].values.reshape(-1, 1)
    y = df["madb_score"].values

    # Plot MADB vs cogent3 scores
    fig = px.scatter(
        df,
        x="cogent3_score",
        y="madb_score",
        color="species",
        color_discrete_map=species_colors,
        symbol="cycle_status",
        symbol_map=symbol_sequence,
        hover_data=["unique_id"]
    )

    # Line of parity
    min_score = min(df["madb_score"].min(), df["cogent3_score"].min())
    max_score = max(df["madb_score"].max(), df["cogent3_score"].max())
    fig.add_trace(go.Scatter(
        x=[min_score, max_score],
        y=[min_score, max_score],
        mode="lines",
        line=dict(color="gray", dash="dash"),
        showlegend=False
    ))

    # Fit linear regression model
    model = LinearRegression().fit(X, y)
    y_pred = model.predict(X)

    # Compute R² and RSS
    r2 = r2_score(y, y_pred)
    rss = ((y - y_pred) ** 2).sum()
    slope = model.coef_[0]

    # Floating annotation
    fig.add_annotation(
        text=f"R² = {r2:.3f}<br>RSS = {rss:.2e}",
        xref="paper", yref="paper",
        x=0.98, y=0.02,
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=14)
    )

    update_layout(fig, in_panel=True, x_title="Needleman-Wunsch SP", y_title="MADB SP")

    fig.show()

    write_pdf = pdf_writer()
    write_pdf(fig, "madb_score_vs_cogent3_score")

graph_accuracy(15)

In [35]:
import pandas
import scipy.stats
from pathlib import Path

def stats_performance(k):
    # Load data
    df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")

    # Filter to desired k-mer size
    df = df[df["kmer_size"] == k].copy()

    # Filter to complete records
    df = df[
        df["cogent3_time"].notnull() &
        df["cogent3_score"].notnull() &
        df["madb_time"].notnull() &
        df["madb_score"].notnull()
    ].copy()

    # Compute performance comparison
    faster_count = (df["madb_time"] < df["cogent3_time"]).sum()
    total = len(df)
    percent_faster = 100 * faster_count / total

    # Speed-up ratios
    speedup_ratios = df["cogent3_time"] / df["madb_time"]
    median_speedup = speedup_ratios.median()
    mean_speedup = speedup_ratios.mean()

    # Wilcoxon test (MADB < Cogent3)
    stat, p_value = scipy.stats.wilcoxon(
        df["madb_time"],
        df["cogent3_time"],
        alternative="less"
    )

    # Construct summary DataFrame
    summary_table = pandas.DataFrame({
        "Metric": [
            "MADB faster than Cogent3 (\\%)",
            "Median speed-up (Cogent3 / MADB)",
            "Mean speed-up (Cogent3 / MADB)",
            "Wilcoxon $p$-value (MADB $<$ Cogent3)"
        ],
        "Value": [
            f"{percent_faster:.1f}\\%",
            f"{median_speedup:.3f}",
            f"{mean_speedup:.3f}",
            f"{p_value:.2g}"
        ]
    })

    # Convert to LaTeX
    latex_table = summary_table.to_latex(
        index=False,
        column_format="lp{8cm}",
        escape=False
    )

    # Write to figures directory
    figures_path = Path("..") / "figures"
    figures_path.mkdir(parents=True, exist_ok=True)
    with open(figures_path / "madb_perf_summary_table.tex", "w") as f:
        f.write(latex_table)

    return summary_table

stats_performance(15)


,Metric,Value
0,MADB faster than Cogent3 (\%),100.0\%
1,Median speed-up (Cogent3 / MADB),1.990
2,Mean speed-up (Cogent3 / MADB),18.597
3,Wilcoxon $p$-value (MADB $<$ Cogent3),4.7e-10
